# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kbhutto256/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

I use **Logistic Regression** as the main model for this lane.

My target is the binary `decline_proxy`: whether a content item loses more than 20% of its first-half March impressions in the second half of March. The five allowed predictors are all available by the March 15 decision point: first-half impressions, clicks, average position, active days, and CTR.

I chose Logistic Regression because the feature set is small and numeric, the outcome is binary, and this week is about beating an honest baseline rather than rewarding complexity. A regularized linear model gives me a readable first learned model and makes it easier to notice if the result is weak or unstable. I also use permutation importance on the held-out set so interpretation is based on measured predictive contribution rather than coefficient size alone.

I **do not** use client/content IDs, second-half metrics, `future_trend_pct`, or any outcome-window feature as predictors. The result is decision-support for observed search-visibility decline in this March slice; it is not a claim about Google's ranking algorithm.



In [43]:
# Setup + rebuild the same March feature frame used in my data contract.
# In Colab, add HF_TOKEN under Secrets. The token is never printed or stored in this notebook.

%pip -q install duckdb huggingface_hub scikit-learn pandas numpy

import os, getpass, duckdb, pandas as pd, numpy as np

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    average_precision_score, roc_auc_score, balanced_accuracy_score,
    precision_score, recall_score, confusion_matrix
)

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

HF_TOKEN = HF_TOKEN or getpass.getpass("Enter Hugging Face READ token (hidden): ")

con = duckdb.connect()
con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN ?)", [HF_TOKEN])

REL = "hf://datasets/FlyRank/internship-warehouse"
DAILY = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"

FEATURE_COLS = [
    "first_half_impressions",
    "first_half_clicks",
    "first_half_avg_position",
    "first_half_active_days",
    "first_half_ctr_pct",
]
TARGET = "decline_proxy"
GROUP = "client_hash_id"

feature_sql = f"""
WITH daily AS (
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COALESCE(gsc_impressions, 0) AS gsc_impressions,
        COALESCE(gsc_clicks, 0) AS gsc_clicks,
        NULLIF(gsc_avg_position, 0) AS gsc_avg_position
    FROM {DAILY}
    WHERE gsc_data_available IS TRUE
),
agg AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(CASE WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
                 THEN gsc_impressions ELSE 0 END) AS first_half_impressions,
        SUM(CASE WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
                 THEN gsc_clicks ELSE 0 END) AS first_half_clicks,
        AVG(CASE WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
                 THEN gsc_avg_position END) AS first_half_avg_position,
        COUNT(DISTINCT CASE WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
                            AND gsc_impressions > 0 THEN report_date END) AS first_half_active_days,
        SUM(CASE WHEN report_date BETWEEN DATE '2026-03-16' AND DATE '2026-03-31'
                 THEN gsc_impressions ELSE 0 END) AS second_half_impressions
    FROM daily
    GROUP BY 1, 2
)
SELECT
    client_hash_id,
    content_hash_id,
    first_half_impressions,
    first_half_clicks,
    first_half_avg_position,
    first_half_active_days,
    100.0 * first_half_clicks / NULLIF(first_half_impressions, 0) AS first_half_ctr_pct,
    CASE
        WHEN first_half_impressions > 0
         AND second_half_impressions < 0.80 * first_half_impressions
        THEN 1 ELSE 0
    END AS decline_proxy
FROM agg
WHERE first_half_impressions > 0
"""

model_df = con.sql(feature_sql).df()

required = set(FEATURE_COLS + [TARGET, GROUP, "content_hash_id"])
missing = sorted(required - set(model_df.columns))
if missing:
    raise ValueError(f"Missing required columns: {missing}")

print(f"Rows in modeling frame: {len(model_df):,}")
print(f"Clients in frame: {model_df[GROUP].nunique():,}")
print(f"Observed decline-proxy rate: {model_df[TARGET].mean():.3f}")
print("Feature columns:", FEATURE_COLS)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows in modeling frame: 151,981
Clients in frame: 44
Observed decline-proxy rate: 0.327
Feature columns: ['first_half_impressions', 'first_half_clicks', 'first_half_avg_position', 'first_half_active_days', 'first_half_ctr_pct']


## 2. Split design

I use a **client-grouped holdout split** rather than a random row split.

Pages from the same client can share measurement history, site-level conditions, tracking patterns, and content-management behavior. If pages from one client appeared in both train and test, the result could look better than it would on a genuinely unseen client. I therefore hold out about 20% of clients with `GroupShuffleSplit`.

The model, the rule baseline, all thresholds used by that baseline, and all metrics are evaluated on this **same held-out test set**. The baseline thresholds are learned only from the training partition, so the test clients do not influence the rule.

This is an honest first validation design for the question I am asking. It is not the final validation audit; next week can test sensitivity across more grouped splits.


In [44]:
# Client-grouped train/test split. IDs are grouping keys only, never predictors.

splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(
    splitter.split(model_df[FEATURE_COLS], model_df[TARGET], groups=model_df[GROUP])
)

train_df = model_df.iloc[train_idx].copy()
test_df = model_df.iloc[test_idx].copy()

train_clients = set(train_df[GROUP])
test_clients = set(test_df[GROUP])

assert train_clients.isdisjoint(test_clients)
assert set(FEATURE_COLS).isdisjoint({GROUP, "content_hash_id"})

print(f"Train rows: {len(train_df):,} | clients: {len(train_clients)}")
print(f"Test rows:  {len(test_df):,} | clients: {len(test_clients)}")
print(f"Train positive rate: {train_df[TARGET].mean():.3f}")
print(f"Test positive rate:  {test_df[TARGET].mean():.3f}")
print("Client overlap:", len(train_clients & test_clients))


Train rows: 138,771 | clients: 35
Test rows:  13,210 | clients: 9
Train positive rate: 0.322
Test positive rate:  0.373
Client overlap: 0


## 3. Train + compare vs my baseline

I compare the learned model with a **transparent Week-4-style rule baseline** on exactly the same held-out client split.

The baseline gives more review priority to pages that, relative to the training distribution, already look fragile at the decision point: weak CTR, worse average position, fewer active days, few clicks, and lower impressions. Its thresholds come only from training data. This keeps the baseline simple and auditable while avoiding test-set tuning.

The primary ranking metric is **Precision@50** because the practical output is a review queue: if I can inspect only 50 pages, how many of those 50 actually meet the decline proxy? I also report PR-AUC, ROC-AUC, balanced accuracy, precision, and recall so one ranking number does not hide weak classification behavior.

I only keep the learned model as an improvement if it beats the baseline on the same test rows and same primary metric. Complexity by itself does not count as progress.


In [45]:
# Transparent baseline + Logistic Regression, evaluated on the SAME held-out clients.

X_train = train_df[FEATURE_COLS]
y_train = train_df[TARGET].astype(int)
X_test = test_df[FEATURE_COLS]
y_test = test_df[TARGET].astype(int)

# ----- Week-4-style transparent rule baseline -----
# Cutoffs are estimated from TRAIN only, then frozen before scoring test.
cut = {
    "ctr_q25": train_df["first_half_ctr_pct"].quantile(0.25),
    "position_q75": train_df["first_half_avg_position"].quantile(0.75),
    "active_q25": train_df["first_half_active_days"].quantile(0.25),
    "clicks_q25": train_df["first_half_clicks"].quantile(0.25),
    "impressions_q25": train_df["first_half_impressions"].quantile(0.25),
}

def baseline_score(frame, thresholds):
    # Higher score = review sooner.
    score = pd.Series(0.0, index=frame.index)
    score += (frame["first_half_ctr_pct"] <= thresholds["ctr_q25"]).astype(float)
    score += (frame["first_half_avg_position"] >= thresholds["position_q75"]).astype(float)
    score += (frame["first_half_active_days"] <= thresholds["active_q25"]).astype(float)
    score += (frame["first_half_clicks"] <= thresholds["clicks_q25"]).astype(float)
    score += (frame["first_half_impressions"] <= thresholds["impressions_q25"]).astype(float)
    return score

baseline_test_score = baseline_score(test_df, cut)

# ----- Learned model -----
model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    )),
])

model.fit(X_train, y_train)
model_prob = model.predict_proba(X_test)[:, 1]
model_pred = (model_prob >= 0.50).astype(int)

# Binary baseline prediction for supporting classification metrics.
# 3+ reason codes = "high priority".
baseline_pred = (baseline_test_score >= 3).astype(int)

def precision_at_k(y_true, score, k=50):
    y_true = np.asarray(y_true)
    score = np.asarray(score)
    if len(y_true) == 0:
        return np.nan
    k = min(k, len(y_true))
    order = np.argsort(-score, kind="mergesort")[:k]
    return float(y_true[order].mean())

def safe_auc(fn, y, score):
    return fn(y, score) if pd.Series(y).nunique() == 2 else np.nan

comparison = pd.DataFrame([
    {
        "method": "Rule baseline",
        "precision_at_50": precision_at_k(y_test, baseline_test_score, 50),
        "pr_auc": safe_auc(average_precision_score, y_test, baseline_test_score),
        "roc_auc": safe_auc(roc_auc_score, y_test, baseline_test_score),
        "balanced_accuracy": balanced_accuracy_score(y_test, baseline_pred),
        "precision": precision_score(y_test, baseline_pred, zero_division=0),
        "recall": recall_score(y_test, baseline_pred, zero_division=0),
    },
    {
        "method": "Logistic Regression",
        "precision_at_50": precision_at_k(y_test, model_prob, 50),
        "pr_auc": safe_auc(average_precision_score, y_test, model_prob),
        "roc_auc": safe_auc(roc_auc_score, y_test, model_prob),
        "balanced_accuracy": balanced_accuracy_score(y_test, model_pred),
        "precision": precision_score(y_test, model_pred, zero_division=0),
        "recall": recall_score(y_test, model_pred, zero_division=0),
    }
]).set_index("method")

display(comparison.round(3))

baseline_p50 = comparison.loc["Rule baseline", "precision_at_50"]
model_p50 = comparison.loc["Logistic Regression", "precision_at_50"]
lift = model_p50 - baseline_p50

print(f"Precision@50 difference (model - baseline): {lift:+.3f}")
if model_p50 > baseline_p50:
    print("Verdict: the learned model beats the transparent baseline on the primary metric.")
elif model_p50 == baseline_p50:
    print("Verdict: no measured Precision@50 improvement over the baseline.")
else:
    print("Verdict: the learned model does NOT beat the baseline; keep the simpler rule for now.")


,precision_at_50,pr_auc,roc_auc,balanced_accuracy,precision,recall
method,,,,,,
Rule baseline,0.54,0.395,0.543,0.523,0.399,0.464
Logistic Regression,0.38,0.395,0.549,0.528,0.397,0.575


Precision@50 difference (model - baseline): -0.160
Verdict: the learned model does NOT beat the baseline; keep the simpler rule for now.


## 4. Errors and interpretation

I interpret the model on the held-out clients rather than on training data.

First, I use **permutation importance with average precision**. A feature is useful here when shuffling it damages held-out ranking quality. This is more directly connected to the task than reading standardized coefficients alone.

Second, I separate **false positives** and **false negatives** at the 0.50 threshold. False positives are pages the model prioritizes as declining even though the proxy says they did not decline; false negatives are actual proxy declines that the model misses. I compare their median feature values with correct predictions to see what kinds of pages are confusing the model.

The error review is descriptive, not causal. For example, if missed declines have strong first-half CTR, that does not mean high CTR causes decline; it means the first-half snapshot did not provide enough signal for those cases.


In [46]:
# Held-out interpretation: permutation importance + compact error analysis.

perm = permutation_importance(
    model,
    X_test,
    y_test,
    scoring="average_precision",
    n_repeats=15,
    random_state=42,
    n_jobs=-1,
)

importance = (
    pd.DataFrame({
        "feature": FEATURE_COLS,
        "mean_importance": perm.importances_mean,
        "std_importance": perm.importances_std,
    })
    .sort_values("mean_importance", ascending=False)
    .reset_index(drop=True)
)

print("Permutation importance on held-out clients (scoring = average precision):")
display(importance.round(4))

error_df = test_df[FEATURE_COLS + [TARGET]].copy()
error_df["prob_decline"] = model_prob
error_df["pred_decline"] = model_pred

error_df["error_type"] = np.select(
    [
        (error_df[TARGET] == 0) & (error_df["pred_decline"] == 1),
        (error_df[TARGET] == 1) & (error_df["pred_decline"] == 0),
        (error_df[TARGET] == 1) & (error_df["pred_decline"] == 1),
    ],
    ["false_positive", "false_negative", "true_positive"],
    default="true_negative",
)

print("Confusion matrix [[TN, FP], [FN, TP]]:")
print(confusion_matrix(y_test, model_pred))
print()
print("Error counts:")
display(error_df["error_type"].value_counts().rename("rows").to_frame())

summary = (
    error_df
    .groupby("error_type")[FEATURE_COLS + ["prob_decline"]]
    .median(numeric_only=True)
    .round(3)
)

print("Median feature profile by prediction outcome:")
display(summary)

# Generate cautious, data-backed sentences from the observed error table.
fp_n = int((error_df["error_type"] == "false_positive").sum())
fn_n = int((error_df["error_type"] == "false_negative").sum())

print()
print("Interpretation notes:")
print(f"- False positives: {fp_n:,}; false negatives: {fn_n:,}.")
if len(importance):
    top = importance.iloc[0]
    print(
        f"- The strongest held-out permutation signal was {top['feature']} "
        f"(mean AP drop {top['mean_importance']:.4f} when shuffled)."
    )
print(
    "- I treat these patterns as directional decision-support only. "
    "They describe this held-out March slice and do not establish why search visibility changed."
)


Permutation importance on held-out clients (scoring = average precision):


,feature,mean_importance,std_importance
0,first_half_clicks,0.0359,0.0017
1,first_half_avg_position,0.0063,0.0018
2,first_half_ctr_pct,-0.0002,0.0005
3,first_half_impressions,-0.0014,0.0022
4,first_half_active_days,-0.0072,0.0043


Confusion matrix [[TN, FP], [FN, TP]]:
[[3973 4305]
 [2094 2838]]

Error counts:


,rows
error_type,
false_positive,4305
true_negative,3973
true_positive,2838
false_negative,2094


Median feature profile by prediction outcome:


,first_half_impressions,first_half_clicks,first_half_avg_position,first_half_active_days,first_half_ctr_pct,prob_decline
error_type,,,,,,
false_negative,268.5,0.0,7.932,15.0,0.000,0.471
false_positive,11.0,0.0,7.618,5.0,0.000,0.572
true_negative,387.0,1.0,7.644,15.0,0.266,0.462
true_positive,11.0,0.0,7.461,5.0,0.000,0.572



Interpretation notes:
- False positives: 4,305; false negatives: 2,094.
- The strongest held-out permutation signal was first_half_clicks (mean AP drop 0.0359 when shuffled).
- I treat these patterns as directional decision-support only. They describe this held-out March slice and do not establish why search visibility changed.


## Self-check

Before I submit, I confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it.
- [ ] The notebook has been run top to bottom in Colab with my Hugging Face `HF_TOKEN`, and all outputs are visible.
- [x] No client names, client URLs, private search queries, or raw IDs are displayed.
- [x] My claims use careful words: observed, measured, directional, decision-support.
- [x] Model and baseline use the same held-out client split and the same primary metric.
- [x] Validation is grouped by client; no client's rows can appear in both train and test.
- [x] Only pre-decision features are used; outcome-window columns and IDs are excluded from predictors.
- [x] The notebook ends with a model-vs-baseline table plus feature/error interpretation.
- [ ] After execution, save/commit this file as `work/notebooks/w05_model.ipynb`, then submit the repository URL on the card.

**Important:** I do not mark the two execution/commit boxes until the Colab run has actually completed with the gated warehouse token and the executed notebook has been saved to GitHub.
